In [1]:
import pandas as pd
from PIL import Image
import io
import requests
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
import json
from google.oauth2 import service_account
from google.auth.transport.requests import Request
import base64

In [2]:
# Load credentials from your service account file
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

In [3]:
pokemon_df = pd.read_csv('data/pokemon_data.csv')
p_images_df = pd.read_csv('data/pokemon_801.csv')

In [4]:
creds.refresh(Request())
token = creds.token

In [5]:
with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

In [6]:
def get_audio(pokemon):

    url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

    payload = {
      "model": "lyria-3-clip-preview",
      "input": [
        {
          "type": "text",
          "text": f"{pokemon} winning theme song"
        }
      ]
    }
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload)
    # print(response)
    # print(response.text)
    # print(response.status_code)
    # print(response.headers)
    data = response.json()
    bytedata = data["outputs"][2]["data"]
    return base64.b64decode(bytedata)

In [7]:
def play_announcement(text):
    try:
        mp3 = get_audio(text)
        display(Audio(mp3, autoplay=True))
    except Exception as e:
        print(f"(Couldn't play audio: {e})")

In [8]:
# pokemon_data is the dataset we've been working with
# pokemon_801 is a dataset with pokemon names and url links of images of each pokemon
# because each dataset has a column called 'name', we can merge the datasets
merged_df = pd.merge(pokemon_df, p_images_df, on="name")
merged_df

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary,image_url
0,"['Overgrow', 'Chlorophyll']",1.00,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,1,65,65,45,grass,poison,6.9,1,0,https://raw.githubusercontent.com/PokeAPI/spri...
1,"['Overgrow', 'Chlorophyll']",1.00,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,2,80,80,60,grass,poison,13.0,1,0,https://raw.githubusercontent.com/PokeAPI/spri...
2,"['Overgrow', 'Chlorophyll']",1.00,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,3,122,120,80,grass,poison,100.0,1,0,https://raw.githubusercontent.com/PokeAPI/spri...
3,"['Blaze', 'Solar Power']",0.50,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,4,60,50,65,fire,NaN,8.5,1,0,https://raw.githubusercontent.com/PokeAPI/spri...
4,"['Blaze', 'Solar Power']",0.50,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,5,80,65,80,fire,NaN,19.0,1,0,https://raw.githubusercontent.com/PokeAPI/spri...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
759,['Beast Boost'],0.25,1.0,0.5,2.0,0.5,1.0,2.0,0.5,1.0,...,797,107,101,61,steel,flying,999.9,7,1,https://raw.githubusercontent.com/PokeAPI/spri...
760,['Beast Boost'],1.00,1.0,0.5,0.5,0.5,2.0,4.0,1.0,1.0,...,798,59,31,109,grass,steel,0.1,7,1,https://raw.githubusercontent.com/PokeAPI/spri...
761,['Beast Boost'],2.00,0.5,2.0,0.5,4.0,2.0,0.5,1.0,0.5,...,799,97,53,43,dark,dragon,888.0,7,1,https://raw.githubusercontent.com/PokeAPI/spri...
762,['Prism Armor'],2.00,2.0,1.0,1.0,1.0,0.5,1.0,1.0,2.0,...,800,127,89,79,psychic,NaN,230.0,7,1,https://raw.githubusercontent.com/PokeAPI/spri...


In [9]:
# now we have all of the original pokemon data, and an image url
# name is a reserved word for a series, so we cannot use dot notation for 'name' and have to use bracket notation!
print(merged_df.iloc[0]["name"])
print(merged_df.iloc[0].image_url)

Bulbasaur
https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/1.png


In [10]:
# decided to save the meowth link for later 
m_url = merged_df.loc[merged_df.name == "Meowth", "image_url"].item()
print(m_url)

https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/52.png


In [11]:
# created a function to convert a url into an image 
# this is doing what a browser does behind the scenes when you go to this url
def get_image_from_url(url):
    response = requests.get(url)
    img = Image.open(io.BytesIO(response.content))
    display(img)

In [12]:
def pokemon_battle():

    # Widgets 
    text_input = widgets.Text(
        placeholder='Pikachu',
        description='Your Pokémon:',
        layout=widgets.Layout(width='300px')
    )
    button = widgets.Button(
        description='Battle!',
        button_style='danger'
    )
    output = widgets.Output()

    # our function - which we will later tie to the battle button
    def on_battle_click(b):
        with output:
            clear_output()

            pokemon_name = text_input.value.strip().capitalize()

            # Look up chosen Pokémon
            match = merged_df[merged_df['name'].str.capitalize() == pokemon_name]

            if match.empty:
                print(f"'{pokemon_name}' not found! Check the spelling and try again.")
                # this ends the function
                return

            my_pokemon = match.iloc[0]
            # get a random pokemon for the opposition
            my_opp    = merged_df.sample().iloc[0]

            my_name  = my_pokemon["name"]
            opp_name = my_opp["name"]

            print(f" I choose you, {my_name}!")
            get_image_from_url(my_pokemon.image_url)
            print(f" It's......{opp_name}!")
            get_image_from_url(my_opp.image_url)

            # Determine winner
            if my_pokemon.hp > my_opp.hp:
                print(my_name)
                get_image_from_url(my_pokemon.image_url)
                play_announcement(f"{my_name} wins!")
            elif my_opp["hp"] > my_pokemon["hp"]:
                print(f"{opp_name} wins!")
                get_image_from_url(my_opp.image_url)
                play_announcement(opp_name)
            else:
                print("It's a tie....meeeooowww")
                get_image_from_url(m_url)
                play_announcement("Meowth")

    # connect our function to the button
    button.on_click(on_battle_click)
    # display the widgets
    display(widgets.VBox([text_input, button, output]))


In [13]:
pokemon_battle()